## L1-Penalized GP with Bilby

*Questions*

- Have to use $\lambda^2 as the rate parameter for tau, but how can I make it conditional? Right now, am drawing a sample manually, but this will mean it won't match up with the sampled values?
- Going to use nu, tau, sigma_noise to calculate values of beta for the log likelihood. 

*To Do*

- set up quest server
- test different samplers
    - 0.2 < acceptance fraction < 0.5
    - look at trace plots and how many samples are needed for convergence
    - try: pyemcee
- run sanity checks
    - take posterior likelihood from BL, check values with likelihood compared to random values


*Notes*

- In BL, using 10,000 iterations with 1000 iterations of burn-in. 
- Gamma prior on lambda squared with r = 1, delta = 1.78
- For beta: using $\nu$ to calculate
- For tau_sq, using $\zeta$ to calculate


### Hidden

\begin{align*}
& p(y, \beta, \tau^2, \sigma^2_N, \sigma^2_{GP}, \ell, \lambda) \\
&= \frac{\exp\!\left(
 -\frac12 (y - X\beta)^\top (K + \sigma^2_N I_n)^{-1} (y - X\beta)
\right)}
{\sqrt{(2\pi)^n |K + \sigma^2_N I_n|}} \\[6pt]
&\quad \times \frac{\exp\!\left(-\frac12 \beta^\top (\sigma^2_N D_\tau)^{-1} \beta \right)}
{(2\pi)^{p/2} |\sigma^2_N D_\tau|^{1/2}} \\[6pt]
&\quad \times \prod_{j=1}^p \frac{\lambda^2}{2} \exp\!\left( -\frac{\lambda^2}{2} \tau_j^2 \right) \\[6pt]
&\quad \times \frac{\beta_{\text{noise}}^{\alpha_{\text{noise}}}}{\Gamma(\alpha_{\text{noise}})}
(\sigma^2_N)^{-\alpha_{\text{noise}}-1} \exp\!\left( -\frac{\beta_{\text{noise}}}{\sigma^2_N} \right) \\[6pt]
&\quad \times \frac{\beta_{\text{GP}}^{\alpha_{\text{GP}}}}{\Gamma(\alpha_{\text{GP}})}
(\sigma^2_{GP})^{-\alpha_{\text{GP}}-1} \exp\!\left( -\frac{\beta_{\text{GP}}}{\sigma^2_{GP}} \right) \\[6pt]
&\quad \times \frac{1}{\ell \, \sigma_\ell \sqrt{2\pi}}
\exp\!\left( -\frac{(\log \ell - \mu_\ell)^2}{2\sigma_\ell^2} \right) \\[6pt]
&\quad \times \frac{b_\lambda^{a_\lambda}}{\Gamma(a_\lambda)} \lambda^{a_\lambda - 1} e^{-b_\lambda \lambda}
\end{align*}

Posterior:
$$p(\beta, \tau^2, \sigma^2_N, \sigma^2_{GP}, \ell, \lambda \mid y) \propto p(y | \beta, \sigma^2_N, \sigma^2_{GP}, \ell) \times p(\beta | \sigma^2_N, \tau^2) \times \prod_{j=1}^p p(\tau_j^2 | \lambda) \times p(\sigma^2_N) \times p(\sigma^2_{GP}) \times p(\ell) \times p(\lambda)$$

In [67]:
## to save time plotting


# fig = plt.Figure() # notice the capital F
# sns.pairplot(.....)
# plt.savefig(.....)
# plt.close()


# pairplot(corner = True) # only plots half of symmetrical plot


### Data

In [68]:
sampler_name = "emcee"
id = 4
numwalkers = 50
numsteps = 1000
a_val = 1.7


In [69]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import bilby
from bilby.core.utils import random
import json
import scipy.special
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# where / how to save files
label = f"{sampler_name}{id}"
outdir = f"{numwalkers}walkers"
bilby.utils.check_directory_exists_and_if_not_mkdir(outdir)
random.seed(123)

In [70]:
### --- to load diabetes data ---
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
X = diabetes.data

selected_features = ['bmi', 'bp', 's1']
X = X[selected_features]

label_names = diabetes.feature_names
y = diabetes.target

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=22)

scaler = StandardScaler()
Xtrain = scaler.fit_transform(Xtrain) # fit and scale training data
Xtest = scaler.transform(Xtest) # scale test data

### MCMC

In [71]:
# helper function to compute the RBF kernel
def rbf_kernel(X, ell, sigma_gp):
    N = X.shape[0]
    K = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            diff = (X[i] - X[j]) / ell
            K[i, j] = sigma_gp**2 * np.exp(-0.5 * np.dot(diff, diff))
    return K

In [72]:
# custom likelihood for penalized GP regression

class PenalizedGPLikelihood(bilby.Likelihood):
    def __init__(self, X, y):
        # store data
        self.X = np.asarray(X)
        self.y = np.asarray(y)

        # define parameters
        parameters = {}

        for i in range(self.X.shape[1]):
            parameters[f"ell{i}"] = None # lengthscales (ells)     
            parameters[f"nu{i}"] = None # nu: random variable for beta calculation
            parameters[f"zeta{i}"] = None # zeta: random variable for tau calculation

        # noise parameters
        parameters["inv_sigma_noise"] = None # remember this is gamma not inverse gamma, so use (alpha, 1/b)
        parameters["inv_sigma_gp"] = None # remember this is gamma not inverse gamma, so use (alpha, 1/b)
        parameters["lambda2"] = None # lambda_squared 

        super().__init__(parameters=parameters)


    def log_likelihood(self):

        # extract parameters
        inv_sigma_noise = self.parameters["inv_sigma_noise"]
        inv_sigma_gp = self.parameters["inv_sigma_gp"]
        ells = np.array([self.parameters[f"ell{i}"] for i in range(self.X.shape[1])])
        lbda2 = self.parameters["lambda2"] # sample lambda squared

        # --- transformers ---
        nus = np.array([self.parameters[f"nu{i}"] for i in range(self.X.shape[1])]) # random var for beta
        zetas = np.array([self.parameters[f"zeta{i}"] for i in range(self.X.shape[1])]) 
        
        # calculate n and p
        n = self.X.shape[0]
        p = self.X.shape[1]

        # --- transform sampled gamma to desired inverse gamma ---
        sigma_gp = 1/inv_sigma_gp if inv_sigma_gp is not None else None
        sigma_noise = 1/inv_sigma_noise if inv_sigma_noise is not None else None

        # --- transform zetas to tau^2 ---
        # each tau^2 \sim expoential(\lambda^2 / 2)
        if (zetas is not None and lbda2 is not None):
            tau_sqs = (2 * zetas) / lbda2

        # --- transform nu to beta ---                   
        if (nus is not None and tau_sqs is not None and sigma_noise is not None):
            betas = nus * np.sqrt(tau_sqs) * sigma_noise 
        
        C = rbf_kernel(self.X, ells, sigma_gp) # calculate covariance matrix C
        residuals = self.y - self.X @ betas  # calculate residuals
        D = np.diag(tau_sqs) # calculate diagonal matrix D

        # --- log likelihood components ---
        log_lik_gp = (
            -0.5 * n * np.log(2 * np.pi) 
            -0.5 * np.linalg.slogdet(C + sigma_noise**2 * np.eye(n))[1]
            -0.5 *(residuals).T @ np.linalg.solve(C + sigma_noise**2 * np.eye(n), residuals))

        log_lik_beta = (
            -0.5 * p * np.log(2 * np.pi)
            -0.5 * np.linalg.slogdet(sigma_noise**2 * D)[1]
            -0.5 * betas.T @ np.linalg.solve(sigma_noise**2 * D, betas))
        
        return log_lik_gp + log_lik_beta


In [73]:
# make priors
priors = dict()

priors["inv_sigma_noise"] = bilby.core.prior.Gamma(1, 1, "inv_sigma_noise")  
priors["inv_sigma_gp"] = bilby.core.prior.Gamma(1, 1, "inv_sigma_gp")  
priors["lambda2"] = bilby.core.prior.Gamma(1.0, 1.78, name="lambda2") # positive

for i in range(Xtrain.shape[1]):
    priors[f"zeta{i}"] = bilby.core.prior.Exponential(1, f"zeta{i}") # exponential with mean 1 = rate 1
    priors[f"ell{i}"] = bilby.core.prior.LogNormal(0, 1, f"ell{i}") # define log-normal priors for each lengthscale
    priors[f"nu{i}"] = bilby.core.prior.Normal(0,1, f"nu{i}") # standard normal for nu

# define the likelihood function that we defined earlier
likelihood = PenalizedGPLikelihood(
    X = Xtrain,
    y = ytrain)

In [61]:
# run MCMC sampler
result = bilby.run_sampler(
    likelihood=likelihood, # likelihood function
    priors=priors, # prior distributions
    sampler=sampler_name, # other options for mcmc are emcee, zeus, , pyemcee, ptemcee, bilby-mcmc 
    nwalkers = numwalkers , # need > 2 x number of parameters
    nsteps = numsteps,
    nburn = 20,
    sampler_kwargs = dict(a=a_val),
    outdir=outdir,
    label=label)

14:38 bilby INFO    : Running for label 'emcee4', output will be saved to '50walkers'
14:38 bilby INFO    : Analysis priors:
14:38 bilby INFO    : inv_sigma_noise=Gamma(k=1, theta=1, name='inv_sigma_noise', latex_label='inv_sigma_noise', unit=None, boundary=None)
14:38 bilby INFO    : inv_sigma_gp=Gamma(k=1, theta=1, name='inv_sigma_gp', latex_label='inv_sigma_gp', unit=None, boundary=None)
14:38 bilby INFO    : lambda2=Gamma(k=1.0, theta=1.78, name='lambda2', latex_label='lambda2', unit=None, boundary=None)
14:38 bilby INFO    : zeta0=Exponential(mu=1, name='zeta0', latex_label='zeta0', unit=None, boundary=None)
14:38 bilby INFO    : ell0=LogNormal(mu=0, sigma=1, name='ell0', latex_label='ell0', unit=None, boundary=None)
14:38 bilby INFO    : nu0=Normal(mu=0, sigma=1, name='nu0', latex_label='nu0', unit=None, boundary=None)
14:38 bilby INFO    : zeta1=Exponential(mu=1, name='zeta1', latex_label='zeta1', unit=None, boundary=None)
14:38 bilby INFO    : ell1=LogNormal(mu=0, sigma=1, name

In [62]:
result.posterior

,inv_sigma_noise,inv_sigma_gp,lambda2,zeta0,ell0,nu0,zeta1,ell1,nu1,zeta2,ell2,nu2,log_likelihood,log_prior
0,0.154151,0.458864,0.872853,2.309956,2.705784,-0.708275,1.238397,1.642403,0.086939,0.982124,3.554954,-0.265775,-25413.789152,-16.196758
1,0.154151,0.458864,0.872853,2.309956,2.705784,-0.708275,1.238397,1.642403,0.086939,0.982124,3.554954,-0.265775,-5813.719565,-14.450795
2,0.154151,0.458864,0.872853,2.309956,2.705784,-0.708275,1.238397,1.642403,0.086939,0.982124,3.554954,-0.265775,-256872.240891,-11.816175
3,0.129264,0.432726,0.831211,2.418435,2.690253,-0.696222,1.257810,1.639523,0.103255,1.010198,3.534100,-0.290186,-94122.922554,-13.715772
4,0.129264,0.432726,0.831211,2.418435,2.690253,-0.696222,1.257810,1.639523,0.103255,1.010198,3.534100,-0.290186,-2755.770460,-15.781181
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48995,0.016135,0.010582,3.805770,0.018937,3.898758,0.756193,0.002530,5.767375,-0.710497,0.043414,3.285278,-0.373621,-1973.918600,-18.672761
48996,0.016135,0.010582,3.805770,0.018937,3.898758,0.756193,0.002530,5.767375,-0.710497,0.043414,3.285278,-0.373621,-2319.740181,-15.350971
48997,0.016135,0.010582,3.805770,0.018937,3.898758,0.756193,0.002530,5.767375,-0.710497,0.043414,3.285278,-0.373621,-1971.387202,-19.187268
48998,0.016135,0.010582,3.805770,0.018937,3.898758,0.756193,0.002530,5.767375,-0.710497,0.043414,3.285278,-0.373621,-2233.986353,-15.519388


In [63]:
# create new dataframe to store results
result_tab = result.posterior.copy()

# calculate tau_sq from samples
for i in range(X.shape[1]):
    zetas = result_tab[f"zeta{i}"].values
    tau_sqs = (2 * zetas) / result_tab["lambda2"].values
    result_tab["lambda"] = np.sqrt(result_tab["lambda2"])
    result_tab[f"tau_sq{i}"] = tau_sqs

# calculate beta from samples
for i in range(X.shape[1]):
    nus = result_tab[f"nu{i}"].values
    result_tab["sigma_noise"] = 1 / result_tab["inv_sigma_noise"].values
    tau_sqs = result_tab[f"tau_sq{i}"].values
    betas = nus * np.sqrt(tau_sqs) * result_tab["sigma_noise"].values
    result_tab[f"beta{i}"] = betas


clean_tab = result_tab.copy()
drop_cols = [f"zeta{i}" for i in range(X.shape[1])] + ["lambda2", "inv_sigma_noise", "inv_sigma_gp"] + [f"nu{i}" for i in range(X.shape[1])]
clean_tab = clean_tab.drop(columns=drop_cols) # drop unneeded columns
clean_tab = clean_tab.reindex(sorted(clean_tab.columns), axis=1) # sort columns alphabetically
    

In [64]:
result.plot_walkers()


In [65]:
# calculate acceptance fraction

import pickle

with open(f"{numwalkers}walkers/{sampler_name}_{sampler_name}{id}/sampler.pickle", "rb") as f:
    sampler = pickle.load(f)

af = sampler.acceptance_fraction
print("Per-walker acceptance fractions:", af)
print("Mean acceptance fraction:", af.mean())


Per-walker acceptance fractions: [0.189 0.062 0.206 0.181 0.098 0.184 0.234 0.208 0.044 0.023 0.206 0.092
 0.209 0.205 0.218 0.267 0.121 0.126 0.186 0.217 0.194 0.188 0.181 0.18
 0.167 0.207 0.18  0.159 0.183 0.182 0.027 0.156 0.203 0.187 0.034 0.119
 0.187 0.09  0.074 0.148 0.034 0.223 0.193 0.222 0.206 0.153 0.102 0.143
 0.035 0.126]
Mean acceptance fraction: 0.15517999999999998


In [66]:
clean_tab.median()

beta0               25.266667
beta1                0.088203
beta2               -2.389541
ell0                 3.032378
ell1                 2.534531
ell2                 3.892930
lambda               1.563648
log_likelihood   -1980.946831
log_prior          -15.958471
sigma_noise         59.957473
tau_sq0              0.531713
tau_sq1              0.484412
tau_sq2              0.587568
dtype: float64